# Bilinmiyor Hat Kodu Tamamlama — Aşama 1 (KESİN)

**Amaç:** Yolcu tablosundaki `GUNCEL_HATKODU = 'Bilinmiyor'` 57,611 kayıt için sefer tablosuyla join ederek gerçek HATKODU'yu bul.

**Mantık (Aşama 1 — KESİN):**
```
JOIN ŞARTI:
  yolcu.KAPINO        = sefer.KAPINO
  yolcu.TARIH         = sefer.TARIH
  yolcu.GECISZAMANI   BETWEEN sefer.BASLANGICZAMANI AND sefer.BITISZAMANI
```
Bu eşleşme **kesindir**: Bir araç bir anda sadece bir seferdedir → HATKODU + YON garantili doğru.

**Beklenen sonuç:** ~43,200 kayıt (sample test sonucu %75 eşleşme) → Bilinmiyor 57K → 14K düşer.

**Çıktı:** `panel_data/bilinmiyor_imputed.json` — `{kayıt_index: HATKODU}` mapping

**NOT:** Orijinal yolcu tablosu DOKUNULMAZ. Sadece bir mapping JSON üretilir.

In [ ]:
import sqlite3
import json
import time
import pandas as pd
from pathlib import Path

DATATHON_DIR = Path(r'c:\Users\asus\Desktop\Datathon')
PANEL_DIR    = DATATHON_DIR / 'panel_data'
DB_PATH      = PANEL_DIR / 'iett_data.db'

conn = sqlite3.connect(DB_PATH)
print(f'DB: {DB_PATH}')

# Toplam Bilinmiyor kayıt sayısı
n_bilinmiyor = pd.read_sql("SELECT COUNT(*) n FROM yolcu WHERE GUNCEL_HATKODU = 'Bilinmiyor'", conn)['n'][0]
print(f'Toplam Bilinmiyor kayit: {n_bilinmiyor:,}')

In [ ]:
# ── Performans için sefer tablosuna geçici index ──
# KAPINO + TARIH üzerinde index varsa join hızlanır
print('Index hazirlanıyor...')
t0 = time.time()
cur = conn.cursor()
cur.execute('CREATE INDEX IF NOT EXISTS idx_sefer_kapino_tarih ON sefer(KAPINO, TARIH)')
conn.commit()
print(f'Index hazir ({time.time()-t0:.1f}sn)')

In [ ]:
# ── AŞAMA 1: KESİN EŞLEŞME ──
# yolcu.ROWID kullanarak her Bilinmiyor kayda unique id veriyoruz
print('Asama 1 SQL JOIN basliyor (kapino + tarih + zaman penceresi)...')
t0 = time.time()

df_imputed = pd.read_sql("""
    SELECT
        y.ROWID as yolcu_rowid,
        y.KAPINO,
        y.TARIH,
        y.GECISZAMANI,
        y.DURAKKODU,
        s.HATKODU      as imputed_hatkodu,
        s.HATCINSI     as imputed_hatcinsi,
        s.HATADI       as imputed_hatadi,
        s.YON          as imputed_yon,
        s.GUZERGAHADI  as imputed_guzergahadi
    FROM yolcu y
    INNER JOIN sefer s
        ON s.KAPINO = y.KAPINO
       AND s.TARIH  = y.TARIH
       AND y.GECISZAMANI BETWEEN s.BASLANGICZAMANI AND s.BITISZAMANI
    WHERE y.GUNCEL_HATKODU = 'Bilinmiyor'
      AND s.HATKODU IS NOT NULL
      AND s.HATKODU != ''
""", conn)

elapsed = time.time() - t0
print(f'JOIN tamamlandi: {len(df_imputed):,} kayit eslesti ({elapsed:.1f}sn = {elapsed/60:.1f}dk)')
print(f'Eslesme orani: %{len(df_imputed)/n_bilinmiyor*100:.1f}')

In [ ]:
# ── Çakışma kontrolü: bir yolcu_rowid birden fazla sefer'le eşleşti mi? ──
dup = df_imputed.duplicated('yolcu_rowid', keep=False)
print(f'Cakisan kayit (>1 sefer): {dup.sum():,}')
if dup.sum() > 0:
    print('Cakisanlardan birinci eslesme aliniyor (ilk sefer kazanir)...')
    df_imputed = df_imputed.drop_duplicates('yolcu_rowid', keep='first')
print(f'Unique eslesme: {len(df_imputed):,}')

In [ ]:
# ── Sonuç istatistikleri ──
print('=== ESLESEN HATLARDA DAGILIM ===')
top_hat = df_imputed['imputed_hatkodu'].value_counts().head(15)
print(top_hat.to_string())
print(f'\nToplam farkli hat: {df_imputed["imputed_hatkodu"].nunique()}')
print(f'Toplam yolculuk eslesme: {len(df_imputed):,}')
print(f'Bilinmiyor kalacak: {n_bilinmiyor - len(df_imputed):,}')
print(f'Iyilesme: %{len(df_imputed)/n_bilinmiyor*100:.1f}')

In [ ]:
# ── JSON çıktı (yolcu_rowid → HATKODU mapping) ──
mapping = {
    str(r['yolcu_rowid']): {
        'hatkodu':    r['imputed_hatkodu'],
        'hatadi':     r['imputed_hatadi'],
        'hatcinsi':   r['imputed_hatcinsi'],
        'yon':        r['imputed_yon'],
        'guzergahadi': r['imputed_guzergahadi'],
    }
    for _, r in df_imputed.iterrows()
}

out = {
    'meta': {
        'kaynak':          'BILINMIYOR_HAT_TAMAMLA.ipynb — Asama 1 (kesin)',
        'mantik':          'yolcu.KAPINO+TARIH+GECISZAMANI BETWEEN sefer.BASLANGIC/BITIS',
        'donem':           '2025-H1',
        'toplam_bilinmiyor': int(n_bilinmiyor),
        'eslesen':         int(len(df_imputed)),
        'eslesme_orani':   round(len(df_imputed) / n_bilinmiyor * 100, 1),
        'bilinmiyor_kalan': int(n_bilinmiyor - len(df_imputed)),
    },
    'top_hatlar': {str(k): int(v) for k, v in top_hat.to_dict().items()},
    'mapping': mapping,
}

OUT = PANEL_DIR / 'bilinmiyor_imputed.json'
with open(OUT, 'w', encoding='utf-8') as f:
    json.dump(out, f, ensure_ascii=False, indent=2)
print(f'JSON yazildi: {OUT} ({OUT.stat().st_size/1024:.0f} KB)')

conn.close()
print('Tamamlandi.')